# State 관리, MessagesState 와 Reducer
- 챗봇·에이전트는 매 턴 메시지가 입니다. 매번 `state["messages"] = [...]` 로 통째 덮어쓰면 이전 메시지가 사라지죠. 
- LangGraph 의 **Reducer (병합 함수)** 가 "이전 값 + 새 값" 을 어떻게 합칠지 결정합니다.

## 환경 준비


```
OPENAI_API_KEY=sk-...

```

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

print("OPENAI_API_KEY:", "있음" if os.getenv("OPENAI_API_KEY") else "없음")

OPENAI_API_KEY: 있음


## 1. 문제, 단순 덮어쓰기는 메시지를 잃는다

In [2]:
%pip install langchain
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, AIMessage

llm = init_chat_model('openai:gpt-5.4-mini')

Note: you may need to restart the kernel to use updated packages.


In [3]:
class NaiveState(TypedDict):
    messages: list                  # 단순 리스트, 매번 덮어씀

def chatbot_node(state: NaiveState) -> dict:
    ai = llm.invoke(state['messages'])
    return {'message':[ai]}

# 그래프 생성
graph = StateGraph(NaiveState)
# 노드 등록
graph.add_node('chatbot', chatbot_node)
# 엣지 연결
graph.add_edge(START, 'chatbot')
graph.add_edge('chatbot', END)
# 컴파일 
app = graph.compile()

In [4]:
result = app.invoke({'messages':[HumanMessage('안녕!! 내 이름은 지유야.')]})

print(len(result['messages']))
for m in result['messages']:
    print(f'{type(m).__name__}{m.content}')

1
HumanMessage안녕!! 내 이름은 지유야.


## 2. 해결, `add_messages` reducer

- State 필드에 `Annotated[list, add_messages]` 로 reducer 를 붙이면 LangGraph 가 **이전 메시지 + 새 메시지** 를 자동으로 이어붙임. 같은 ID 메시지는 자동 dedup.

### `Annotated[list, add_messages]` 가 하는 일

- `Annotated` 는 "타입 + 메타데이터" 를 동시에 적는 파이썬 문법. 여기서 메타데이터 자리에 들어간 `add_messages` 가 reducer.

- 노드가 `{"messages": [새 메시지]}` 를 반환하면
- LangGraph 가 **기존 messages 리스트와 새 리스트를 합쳐서** 새 state 를 만듦
- reducer 없으면 단순 덮어쓰기 (1번 예시 처럼 이전 메시지가 사라짐)

In [5]:
from typing import Annotated
from langgraph.graph.message import add_messages


class ChatState(TypedDict):
    messages: Annotated[list, add_messages]     


def chatbot_node(state: ChatState) -> dict:
    ai = llm.invoke(state["messages"])
    return {"messages": [ai]}               

In [6]:
# 그래프 생성
graph = StateGraph(ChatState)
# 노드 등록
graph.add_node('chatbot', chatbot_node)
# 엣지 연결
graph.add_edge(START, 'chatbot')
graph.add_edge('chatbot', END)
# 컴파일
app = graph.compile()


In [7]:
result = app.invoke({"messages": [HumanMessage("안녕!! 내 이름은 도훈이야.")]})

print("최종 messages 길이:", len(result["messages"]))
for m in result["messages"]:
    print(f"  [{type(m).__name__}] {m.content[:60]}")

최종 messages 길이: 2
  [HumanMessage] 안녕!! 내 이름은 도훈이야.
  [AIMessage] 안녕 도훈아!! 반가워 😊  
무엇을 도와줄까?


## 3. `MessagesState`, 자주 쓰는 패턴 한 줄 단축

- `messages` 필드를 자주 쓰니까 LangGraph 가 **`MessagesState`** 라는 미리 만든 클래스를 제공. 위 ChatState 와 동일.

In [8]:
from langgraph.graph import MessagesState

def chatbot_node(state: MessagesState) -> dict:
    ai = llm.invoke(state['messages'])
    return {'messages': [ai]}

# 그래프 생성
graph = StateGraph(ChatState)
# 노드 등록
graph.add_node('chatbot', chatbot_node)
# 엣지 연결
graph.add_edge(START, 'chatbot')
graph.add_edge('chatbot', END)
# 컴파일
app = graph.compile()

result = app.invoke({'messages': [HumanMessage('안녕!! 내 이름은 누구야.')]})
result['messages'][-1].content

'안녕! 나는 지금 **네 이름을 알 수 없어**.  \n네가 직접 알려주면 내가 기억해서 불러줄 수 있어!'

## 4. 멀티턴, 같은 그래프를 반복 호출

- 각 호출 결과를 다음 호출의 입력으로 넘기면 대화 누적.

### 4.1. 대화 저장하고 이어하기, `InMemorySaver` + `thread_id`

- `global state` 변수 트릭은 한 노트북 안에서만 됩니다. 진짜 챗봇은 **사용자별 / 세션별로 대화를 따로 저장** 해야 하죠. LangGraph 의 **checkpointer** 가 이걸 자동으로 해줍니다.

- `compile(checkpointer=...)` 로 그래프에 저장소 붙임
- 호출 시 `config={"configurable": {"thread_id": "user-123"}}` 로 세션 ID 지정
- 같은 thread_id 로 다시 호출하면 이전 대화가 자동으로 복원

In [9]:
from langgraph.checkpoint.memory import InMemorySaver

# 인메모리 저장소(프로세스가 살아있는 동안만). 운영에서는 PostgreSQL 등으로 교체.
checkpointer = InMemorySaver()

In [10]:
# 같은 그래프를 checkpointer와 함께 다시 컴파일
graph = StateGraph(MessagesState)
graph.add_node('chatbot', chatbot_node)
graph.add_edge(START, 'chatbot')
graph.add_edge('chatbot', END)

app = graph.compile(checkpointer=checkpointer)

In [11]:
# 세션 ID로 사용자 구분
config_user_a = {'configurable': {'thread_id':'user-a'}}
config_user_b = {'configurable': {'thread_id':'user-b'}}

In [12]:
# user a 자기소개
app.invoke({'messages': [HumanMessage('안녕! 내 이름은 도훈이야.')]}, config=config_user_a)

# user b 자기소개
app.invoke({'messages': [HumanMessage('안녕! 내 이름은 진남이야.')]}, config=config_user_b)

# user a가 다시 옴 -> 이전 대화 자동 복원
result_a = app.invoke({'messages': [HumanMessage('내 이름이 뭐라고?')]}, config=config_user_a)
print('user a:', result_a['messages'][-1].content)

# user b가 다시 옴 -> 이전 대화 자동 복원
result_b = app.invoke({'messages': [HumanMessage('내 이름이 뭐라고?')]}, config=config_user_b)
print('user b:', result_b['messages'][-1].content)

user a: 당신의 이름은 도훈이예요.
user b: 진남님이라고 하셨어요.


### 4.2. `app.stream` 으로 노드별 진행 보기

- `invoke` 는 최종 결과만 돌려주지만, `stream` 으로 받으면 **노드 하나 끝날 때마다 중간 결과**가 흘러옵니다. 디버깅·UI 진행바에 활용.

In [13]:
for chunk in app.stream(
    {"messages": [HumanMessage("에이전트를 배우는 중인데 학습 순서 추천해주세요.")]},
    config=config_user_a,
    stream_mode="updates",
):
    for node_name, payload in chunk.items():
        last = payload["messages"][-1]
        print(f"[{node_name}] {type(last).__name__}: {last.content}")

[chatbot] AIMessage: 물론이죠, 도훈이님. 에이전트를 배우는 순서를 **“기초 → 도구 사용 → 계획/실행 → 메모리 → 멀티에이전트 → 평가/운영”** 흐름으로 잡으면 이해가 훨씬 편합니다.

## 추천 학습 순서

### 1) 에이전트의 기본 개념
먼저 아래 개념부터 익히세요.
- **LLM**: 문장을 이해하고 생성하는 모델
- **Agent**: 단순히 답만 하는 게 아니라,  
  **목표를 세분화하고, 도구를 쓰고, 결과를 확인하며 행동하는 시스템**
- **Tool**: 검색, 코드 실행, DB 조회, API 호출 같은 외부 기능
- **Memory**: 이전 대화나 작업 결과를 저장해서 다음에 활용
- **Planner / Executor**: 계획을 세우는 부분과 실행하는 부분

이 단계에서는 “에이전트가 왜 필요한지”를 이해하는 게 핵심입니다.

---

### 2) 프롬프트와 함수 호출
에이전트는 프롬프트만 잘 써서는 부족하고, **도구를 호출하는 구조**를 알아야 합니다.

배울 것:
- 프롬프트 설계 기본
- JSON 형태 출력
- 함수 호출(Function Calling)
- 구조화된 출력
- 도구 선택 기준

추천 목표:
- “질문에 답하기” → “필요하면 검색 도구를 호출하기”로 확장

---

### 3) 간단한 단일 에이전트 만들기
처음부터 복잡한 멀티에이전트보다, **하나의 에이전트가 여러 도구를 쓰는 구조**부터 만들어보세요.

예시:
- 날씨 조회
- 간단한 웹 검색
- 계산기 호출
- 파일 읽기/쓰기

여기서 익힐 것:
- ReAct 스타일 사고-행동 루프
- tool calling
- 실패 처리
- 반복 실행 횟수 제한

---

### 4) 계획과 실행 분리
조금 익숙해지면 **Plan → Act → Reflect** 구조를 배우면 좋습니다.

학습 포인트:
- 작업을 작은 단계로 나누기
- 계획을 먼저 만들고 실행하기
- 중간 결과를 보고 수정하기
- 자기 점검(Reflection)

이 단계가 되면

In [14]:
state = {"messages": []}

def chat(user_text):
    global state
    state["messages"].append(HumanMessage(user_text))
    state = app.invoke(state, config=config_user_a)         
    return state["messages"][-1].content


print("Q1:", "내 이름 철수야. 인공지능을 배우는 학생이야.")
print("A1:", chat("내 이름은 철수야. 인공지능을 배우는 학생이야."))
print()
print("Q2:", "내가 누구라고?")
print("A2:", chat("내가 누구라고?"))  

Q1: 내 이름 철수야. 인공지능을 배우는 학생이야.
A1: 알겠습니다, 철수님.  
인공지능을 배우는 학생이라면 에이전트 학습은 아래 순서가 특히 좋아요.

## 추천 순서
1. **머신러닝/딥러닝 기초**
   - 지도학습, 비지도학습, 과적합
   - 신경망, 손실함수, 역전파

2. **LLM 기초**
   - 토크나이저
   - 임베딩
   - 프롬프트
   - 추론 방식

3. **에이전트 기본 개념**
   - 목표 설정
   - 도구 사용
   - 계획/실행/반성
   - 메모리

4. **함수 호출과 툴 사용**
   - JSON 출력
   - API 호출
   - 검색/계산/DB 연결

5. **단일 에이전트 구현**
   - ReAct 패턴
   - 반복 루프
   - 실패 처리

6. **워크플로우 기반 에이전트**
   - 상태 머신
   - 조건 분기
   - 재시도
   - 인간 승인

7. **멀티에이전트**
   - 역할 분담
   - 협업
   - 조율자 패턴

8. **평가와 배포**
   - 정확도
   - 응답 속도
   - 비용
   - 로그/모니터링

## 학생에게 특히 중요한 공부법
- **이론 30% + 실습 70%**
- 작은 프로젝트를 빨리 만들어보기
- “에이전트가 언제 실패하는지”를 기록하기
- 논문보다 먼저 **동작하는 코드**를 만져보기

## 추천 미니 프로젝트
- Q&A 챗봇
- 웹 검색 에이전트
- 문서 요약 에이전트
- 일정 관리 에이전트
- 간단한 연구 보조 에이전트

원하시면 제가 다음으로  
**“철수님용 2주/4주 학습 계획”**이나  
**“파이썬으로 에이전트 입문하는 방법”**을 바로 짜드릴게요.

Q2: 내가 누구라고?
A2: 당신은 **철수님**이고, **인공지능을 배우는 학생**이라고 말씀하셨어요.


## 5. (심화) 커스텀 State + 여러 필드 reducer

- `messages` 외에 다른 필드도 reducer 를 지정합니다. 예: 점수 누적.

In [15]:
from typing import Annotated
from operator import add                        # 숫자 누적용 reducer


class QuizState(TypedDict):
    messages: Annotated[list, add_messages]
    score: Annotated[int, add]                  # 점수 : 노드가 반환한 값을 기존 더함  


def grade_node(state: QuizState) -> dict:
    """마지막 답이 '예' 면 점수 +1."""
    last = state["messages"][-1].content.strip()
    return {"score": 1 if "예" in last else 0}


graph = StateGraph(QuizState)
graph.add_node("grade", grade_node)
graph.add_edge(START, "grade")
graph.add_edge("grade", END)
app_quiz = graph.compile()

In [16]:
# 초기 state를 만들기. 첫 번 째 호출. score는 0점에서부터 시작
state = {'messages': [HumanMessage("예")], "score":0}
state = app_quiz.invoke(state)
print(state['score'])

1


In [17]:
state['messages'].append(HumanMessage("예"))
state = app_quiz.invoke(state)
print(state['score'])

2


## 6. (심화) Pydantic State

- TypedDict 대신 Pydantic 모델도 가능. 필드 검증·기본값 지원.

In [18]:
from pydantic import BaseModel, Field

In [19]:
# BaseModel
class PydState(BaseModel):
    messages: Annotated[list, add_messages] = Field(default_factory=list)
    turn: Annotated[int, add] = 0

def turn_counter(state: PydState) -> dict:
    return {"turn":1}       # 매번 + 1

graph = StateGraph(PydState)
graph.add_node("count", turn_counter)
graph.add_edge(START, "count")
graph.add_edge("count", END)
app_p = graph.compile()

state = PydState()
state = app_p.invoke(state)
print("1회 후 turn:", state["turn"])
state = app_p.invoke(state)
print("2회 후 turn:", state["turn"])

1회 후 turn: 1
2회 후 turn: 2


## 7. 정리

| 패턴 | 언제 |
|---|---|
| `TypedDict + Annotated[list, add_messages]` | 채팅·에이전트 표준 |
| `MessagesState` | 위와 동일한 한 줄 단축 |
| `Annotated[int, add]` 같은 다른 reducer | 점수·카운트·합산 필드 |
| `Pydantic BaseModel` | 필드 검증·기본값 필요할 때 |


## [실습]

1. `MessagesState` 로 3턴 대화 (내 이름 -> 문의한 주문번호 -> 해당 문의 응대 방향) 가 모두 기억되는지 확인.
2. `chatbot_node` 의 system 프롬프트를 "고객지원 매니저 톤으로 답해" 로 바꿔보기.
3. `QuizState` 의 `score` 외에 `correct_questions: Annotated[list, add]` 필드 추가해서 맞춘 질문 누적.
4. Pydantic State 에 `remaining_sla_minutes: int = 100` 추가하고 매 노드에서 -10 씩 감소시키는 reducer 만들기 (`lambda old, new: old + new` 직접).
5. `messages` 가 50개 넘으면 자동으로 오래된 거 잘라내는 wrapper 노드 추가.


In [21]:
def chatbot_node(state: NaiveState) -> dict:
    ai = llm.invoke(state['messages'])
    return {'message': [ai]}

graph2 = StateGraph(MessagesState)
graph2.add_node('chatbot', chatbot_node)
graph2.add_edge(START, 'chatbot')
graph2.add_edge('chatbot', END)

app2 = graph2.compile(checkpointer=checkpointer)

In [24]:
# 사용자 구분(config)
config_user_c = {'configurable' : {'thread_id':'user-c'}}

state2 = {'messages': []}

def chat(user_text):
    global state2
    # 1. 사용자 입력을 HumanMessage 객체로 만들어 메시지 목록에 추가합니다.
    state2['messages'].append(HumanMessage(content=user_text))
    # 2. 업데이트된 상태로 AI 앱을 호출합니다.
    state2 = app2.invoke(state2, config=config_user_c)
    # 3. 앱 실행 후 추가된 마지막 메시지(AI의 응답)를 반환합니다.
    return state2['messages'][-1].content

print('Q1:', '내 이름은 코난. 탐정이 될 계획이지.')
print('A1:', chat('내 이름은 코난. 탐정이 될 계획이지.'))
print()
print('Q2:', '내가 누구라고?')
print('A2:', chat('내가 누구라고?'))
print()
print('Q3:', '내가 누구라고? 그리고 앞으로의 탐정사무소 채용 계획을 설명해줘!')
print('A3:', chat('내가 누구라고? 그리고 앞으로의 탐정사무소 채용 계획을 설명해줘!'))

Q1: 내 이름은 코난. 탐정이 될 계획이지.
A1: 내 이름은 코난. 탐정이 될 계획이지.

Q2: 내가 누구라고?
A2: 내가 누구라고?

Q3: 내가 누구라고? 그리고 앞으로의 탐정사무소 채용 계획을 설명해줘!
A3: 내가 누구라고? 그리고 앞으로의 탐정사무소 채용 계획을 설명해줘!
